In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=cabcee89-40d9-456c-99f0-457a2e155b5a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d195026fe7830ca8f9ce980724b424714f17be8e76da4878dc23c9ee4ec1c3f9




Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection"

Repository Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection initialized!

In [5]:
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

print(test_transaction.shape)
print(test_identity.shape)

(506691, 393)
(141907, 41)


# Merge the data

In [6]:
df_test = test_transaction.merge(test_identity, on='TransactionID', how='left')
print(f"Merged shape: {df_test.shape}")

transaction_ids = df_test['TransactionID']

X_test = df_test.drop(columns=['TransactionID'])

print(f"X_test shape: {X_test.shape}")

Merged shape: (506691, 433)
X_test shape: (506691, 432)


# Load model from registry and predict

In [8]:
model_uri = "models:/IEEE_Fraud_Detection_Best_Model/2"
pipeline = mlflow.sklearn.load_model(model_uri)

print("Model loaded successfully!")

y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Predictions shape: {y_pred_proba.shape}")
print(f"Sample predictions: {y_pred_proba[:5]}")

Model loaded successfully!
Predictions shape: (506691,)
Sample predictions: [0.02413507 0.04298331 0.01415937 0.02046026 0.05558872]


# Generate submission.csv

In [9]:
submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud': y_pred_proba
})

submission.to_csv('submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
print(submission.head())

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.024135
1        3663550  0.042983
2        3663551  0.014159
3        3663552  0.020460
4        3663553  0.055589
